# Pronósticos de salarios en México - ARIMA y SARIMA

**Modelos Avanzados de Analítica de Datos**

Pronósticos mensuales de 2022, 2023 y 2024.

### 1. Importar librerías y datos

In [ ]:
#Instalamos la librería
!pip -q install pmdarima

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import statsmodels.api as sm
from statsmodels.graphics.tsaplots import plot_acf
from statsmodels.tsa.stattools import adfuller
from statsmodels.tsa.holtwinters import ExponentialSmoothing
from pmdarima import auto_arima
from sklearn.metrics import mean_squared_error, mean_absolute_percentage_error
from IPython.display import display

In [ ]:
#Importar la serie
ts = pd.read_excel('Ejercicio_Salarios.xlsx')
print(ts.shape)
ts.head()

In [ ]:
#Que tipo de datos tenemos?
ts.dtypes

In [ ]:
#Convertimos a formato fecha el tiempo
ts['Periodo'] = pd.to_datetime(ts['Periodo'], format='%Y/%m')
#Convertimos el tiempo en índice
ts = ts.set_index('Periodo').sort_index()
ts = ts.asfreq('MS')
ts.head()

In [ ]:
# Graficar la serie de tiempo
plt.figure(figsize=(14, 4))
plt.plot(ts['Salarios MXP por hora trabajada'])
plt.xticks(rotation=45)
plt.xlabel('Tiempo')
plt.ylabel('Salarios MXP por hora')
plt.title('Remuneración por hora trabajada en México')
plt.show()

### 2. Revisar la serie

In [ ]:
#Obtener correlograma
plot_acf(ts['Salarios MXP por hora trabajada'], lags=36)
plt.xlabel('Lags o coeficientes')
plt.title('Correlograma')
plt.show()

In [ ]:
#Definimos la prueba
adf_test = adfuller(ts['Salarios MXP por hora trabajada'], autolag='AIC')
p_value = adf_test[1]
print('p-value:', p_value)
if p_value < 0.05:
    print('La serie es estacionaria')
else:
    print('La serie no es estacionaria')

### 3. Ajustar ARIMA y SARIMA

Se utiliza el 80% de los datos para entrenamiento y el 20% para evaluar los pronósticos. Primero se prueba ARIMA y después el modelo con estacionalidad de 12 meses.

In [ ]:
# Dividir la serie en 80% entrenamiento y 20% prueba
n_train = int(len(ts) * 0.8)
train = ts['Salarios MXP por hora trabajada'].iloc[:n_train]
test = ts['Salarios MXP por hora trabajada'].iloc[n_train:]
train.shape, test.shape

In [ ]:
# Ajustar Auto ARIMA solo con los datos de entrenamiento
model_train_arima = auto_arima(train, seasonal=False, stepwise=True, suppress_warnings=True)

# Ajustar el modelo con estacionalidad
L = 12
model_train = auto_arima(train, m=L, seasonal=True, stepwise=True, suppress_warnings=True)

print('ARIMA:', model_train_arima.order)
print('SARIMA:', model_train.order, model_train.seasonal_order)
print(model_train.summary())

In [ ]:
# Mostrar los coeficientes del modelo
print('Modelo:', model_train.order, model_train.seasonal_order)
print(model_train.arima_res_.params)
# La ecuación incluye los términos AR, MA y los términos estacionales

### 4. Evaluación de ARIMA y SARIMA

In [ ]:
# Obtener los valores ajustados de entrenamiento
y_pred_train = model_train.predict_in_sample()

# Pronosticar los valores del conjunto de prueba
y_pred_test = model_train.predict(n_periods=len(test))
y_pred_test = pd.Series(np.asarray(y_pred_test), index=test.index)

# Comparar también con ARIMA sin estacionalidad
y_pred_test_arima = pd.Series(np.asarray(model_train_arima.predict(n_periods=len(test))), index=test.index)

# Obtenemos las métricas de error
mse_test = mean_squared_error(test, y_pred_test)
rmse_test = np.sqrt(mse_test)
mape_test = mean_absolute_percentage_error(test, y_pred_test)*100
mpe_test = np.mean((test - y_pred_test)/test)*100

mse_arima = mean_squared_error(test, y_pred_test_arima)
rmse_arima = np.sqrt(mse_arima)
mape_arima = mean_absolute_percentage_error(test, y_pred_test_arima)*100
mpe_arima = np.mean((test - y_pred_test_arima)/test)*100

metricas_sarima = {'MSE':mse_test, 'RMSE':rmse_test, 'MAPE (%)':mape_test, 'MPE (%)':mpe_test}
metricas_arima = {'MSE':mse_arima, 'RMSE':rmse_arima, 'MAPE (%)':mape_arima, 'MPE (%)':mpe_arima}

print('Resultados SARIMA:')
print(pd.Series(metricas_sarima).round(3))
print('Resultados ARIMA:')
print(pd.Series(metricas_arima).round(3))
print('MPE positivo: el modelo subestima; MPE negativo: sobreestima')

In [ ]:
# Gráfico de la serie real, ajustada y pronosticada en prueba
plt.figure(figsize=(14, 4))
plt.plot(train.index, train, label='Train', color='black')
plt.plot(test.index, test, label='Test', color='green')
plt.plot(test.index, y_pred_test, label='SARIMA', color='blue')
plt.plot(test.index, y_pred_test_arima, label='ARIMA', color='red')
plt.legend()
plt.title('Pronósticos en datos de prueba')
plt.show()

### 5. Pronósticos de 2022 a 2024

Se ajusta el modelo estacional con las 120 observaciones y se calculan los 36 meses siguientes.

In [ ]:
# Ajustar el modelo Auto ARIMA con toda la serie
L = 12
model = auto_arima(ts['Salarios MXP por hora trabajada'], m=L, seasonal=True,
                   stepwise=True, suppress_warnings=True)
print('Modelo final:', model.order, model.seasonal_order)
print(model.summary())
print('Coeficientes finales:')
print(model.arima_res_.params)

In [ ]:
# Ecuación del modelo final
coef = model.arima_res_.params
ma1 = coef['ma.L1']
ma12 = coef['ma.S.L12']
intercepto = coef['intercept'] if 'intercept' in coef.index else 0
sigma2 = coef['sigma2']

ecuacion = (f'(1-B)(1-B^12)Yt = {intercepto:.6f} '
            f'+ (1 {ma1:+.6f}B)(1 {ma12:+.6f}B^12)et')
print('Modelo final: SARIMA', model.order, model.seasonal_order)
print('Ecuación estimada:')
print(ecuacion)
print('Varianza del error:', round(sigma2, 6))
print('B es el operador de rezago; et representa el error del modelo.')


In [ ]:
# Obtener los valores ajustados
y_fit = pd.Series(np.asarray(model.predict_in_sample()), index=ts.index)

# Pronosticar futuros valores
n_forecast = 36
future_dates = pd.date_range(start=ts.index[-1] + pd.offsets.MonthBegin(1),
                             periods=n_forecast, freq='MS')
y_pred = model.predict(n_periods=n_forecast)
y_pred = pd.Series(np.asarray(y_pred), index=future_dates)

print('Primeros 6 pronósticos:')
print(y_pred.head(6).round(2))

In [ ]:
# Gráfico de la serie histórica, valores ajustados y pronósticos
plt.figure(figsize=(14, 4))
plt.plot(ts.index, ts['Salarios MXP por hora trabajada'], label='y', color='black')
plt.plot(y_fit.index[1:], y_fit.iloc[1:], label='y_fit', color='red')
plt.plot(y_pred.index, y_pred, label='y_pred', color='blue')
plt.axvline(x=y_pred.index[0], color='gray', linestyle='--')
plt.title('Pronósticos 2022-2024')
plt.legend()
plt.xlabel('Periodo')
plt.ylabel('MXP por hora')
plt.show()

In [ ]:
# Promedio de pronósticos por año
promedios_anuales = y_pred.resample('YE').mean()
print('Promedios anuales:')
print(promedios_anuales.round(2))
print('Cambio del promedio anual de 2022 a 2024:',
      round(promedios_anuales.iloc[-1] - promedios_anuales.iloc[0], 2))
print('Variación porcentual del promedio anual:',
      round((promedios_anuales.iloc[-1]/promedios_anuales.iloc[0] - 1)*100, 2), '%')
print('Los valores de diciembre son mayores por la estacionalidad; no deben compararse directamente con enero.')


### 6. Comparación con Winters, AD y MD

In [ ]:
# Holt-Winters - modelo aditivo y multiplicativo
L = 12
resultados = {'ARIMA': metricas_arima, 'SARIMA': metricas_sarima}
pronosticos = pd.DataFrame({'SARIMA': y_pred})

for tipo in ['add', 'mul']:
    #Definir el modelo
    model_train_w = ExponentialSmoothing(train, trend='add', seasonal=tipo, seasonal_periods=L)
    fit_train_w = model_train_w.fit()
    y_pred_test_w = fit_train_w.forecast(steps=len(test))

    # Obtenemos las metricas de error
    mse = mean_squared_error(test, y_pred_test_w)
    rmse = np.sqrt(mse)
    mape = mean_absolute_percentage_error(test, y_pred_test_w)*100
    mpe = np.mean((test - y_pred_test_w)/test)*100
    nombre = 'Winters aditivo' if tipo == 'add' else 'Winters multiplicativo'
    resultados[nombre] = {'MSE':mse, 'RMSE':rmse, 'MAPE (%)':mape, 'MPE (%)':mpe}

    model_w = ExponentialSmoothing(ts['Salarios MXP por hora trabajada'],
                                   trend='add', seasonal=tipo, seasonal_periods=L)
    fit_w = model_w.fit()
    pronosticos[nombre] = np.asarray(fit_w.forecast(steps=n_forecast))

print(pd.DataFrame(resultados).T.round(3))

In [ ]:
# Pronosticar la tendencia y estacionalidad con descomposición
# Mismo procedimiento para AD y MD
for tipo in ['additive', 'multiplicative']:
    # Ajustar la descomposición solo con entrenamiento
    model_train_d = sm.tsa.seasonal_decompose(train, model=tipo, period=L)
    trend_train = model_train_d.trend.dropna()
    x_train = np.arange(len(train))[model_train_d.trend.notna()]
    coef_train = np.polyfit(x_train, trend_train.values, 1)
    future_index = np.arange(len(train), len(train) + len(test))
    trend_forecast_test = np.polyval(coef_train, future_index)
    seasonal_forecast_test = np.tile(model_train_d.seasonal.iloc[-L:].to_numpy(),
                                    len(test)//L + 1)[:len(test)]
    if tipo == 'additive':
        pred_test_d = trend_forecast_test + seasonal_forecast_test
        nombre = 'AD'
    else:
        pred_test_d = trend_forecast_test * seasonal_forecast_test
        nombre = 'MD'
    pred_test_d = pd.Series(pred_test_d, index=test.index)

    mse = mean_squared_error(test, pred_test_d)
    rmse = np.sqrt(mse)
    mape = mean_absolute_percentage_error(test, pred_test_d)*100
    mpe = np.mean((test - pred_test_d)/test)*100
    resultados[nombre] = {'MSE':mse, 'RMSE':rmse, 'MAPE (%)':mape, 'MPE (%)':mpe}

    # Pronosticar futuros valores con toda la serie
    model_d = sm.tsa.seasonal_decompose(ts['Salarios MXP por hora trabajada'],
                                        model=tipo, period=L)
    trend = model_d.trend.dropna()
    x = np.arange(len(ts))[model_d.trend.notna()]
    coef = np.polyfit(x, trend.values, 1)
    future_index = np.arange(len(ts), len(ts) + n_forecast)
    trend_forecast = np.polyval(coef, future_index)
    seasonal_forecast = np.tile(model_d.seasonal.iloc[-L:].to_numpy(),
                                n_forecast//L + 1)[:n_forecast]
    if tipo == 'additive':
        pronosticos[nombre] = trend_forecast + seasonal_forecast
    else:
        pronosticos[nombre] = trend_forecast * seasonal_forecast

In [ ]:
# Comparar los errores de los modelos
comparacion = pd.DataFrame(resultados).T
print(comparacion.round(3))
mejor = comparacion['RMSE'].idxmin()
print('Modelo con menor RMSE en test:', mejor)
print('MPE positivo: el modelo subestima; MPE negativo: sobreestima')

In [ ]:
# Mostrar pronósticos mensuales para los próximos 3 años
print(pronosticos.round(2).to_string())

# Gráfico de la serie histórica y pronósticos
plt.figure(figsize=(14, 4))
plt.plot(ts.index, ts['Salarios MXP por hora trabajada'], label='Real', color='black')
for nombre in pronosticos.columns:
    plt.plot(pronosticos.index, pronosticos[nombre], label=nombre)
plt.legend()
plt.title('Comparación de pronósticos 2022-2024')
plt.xlabel('Periodo')
plt.ylabel('MXP por hora')
plt.show()

### 7. Respuestas y resultados en Excel

In [ ]:
# Respuestas de la actividad
print('1. Modelo final: SARIMA', model.order, model.seasonal_order)
print('2. Ecuación:', ecuacion)
print('   Varianza residual:', round(sigma2, 6))
print('3. Primeros seis pronósticos:')
print(y_pred.head(6).round(2))
print('4. Métricas en test:')
print(pd.Series(metricas_sarima).round(3))
print('   El MAPE indica el error porcentual absoluto medio.')
print('   El MPE positivo indica una ligera subestimación promedio.')
print('5. Gráfica de valores reales, ajustados y pronósticos: ver sección 5.')
print('6. Promedios anuales:')
print(promedios_anuales.round(2))
print('   El promedio previsto crece a lo largo de los tres años.')
print('7. Modelo con menor RMSE en test:', mejor)
print('   Se compararon los modelos con los mismos 24 meses de prueba.')


### Respuestas finales

El pronóstico final se realizó con SARIMA estacional de 12 meses. También se evaluó un ARIMA sin componente estacional para poder comparar ambos métodos.

Los promedios anuales del salario pronosticado muestran un incremento entre 2022 y 2024. Los valores de diciembre son más altos por el patrón estacional, por eso para evaluar la evolución conviene comparar los promedios anuales.

El mejor modelo en el periodo de prueba se determina usando el RMSE y el MAPE de los mismos 24 meses. El MPE muestra el sentido del error, pero no demuestra por sí solo ausencia de sesgo. Los pronósticos son estimaciones, no valores observados.


In [ ]:
# Guardar los resultados en Excel
validacion = pd.DataFrame({'Real':test, 'ARIMA':y_pred_test_arima, 'SARIMA':y_pred_test})
coeficientes = model.arima_res_.params.to_frame('Coeficiente')
respuestas = pd.DataFrame([
    ['1. Modelo final', 'SARIMA' + str(model.order) + str(model.seasonal_order)],
    ['2. Ecuación', ecuacion],
    ['2. Varianza residual', round(sigma2, 6)],
    ['3. Primeros 6 meses', '; '.join(f'{fecha:%Y-%m}: {valor:.2f}' for fecha, valor in y_pred.head(6).items())],
    ['4. MSE test', round(mse_test, 3)],
    ['4. RMSE test', round(rmse_test, 3)],
    ['4. MAPE test (%)', round(mape_test, 3)],
    ['4. MPE test (%)', round(mpe_test, 3)],
    ['4. Interpretación', 'El MAPE es el error porcentual absoluto medio y el MPE positivo señala subestimación promedio.'],
    ['5. Gráfica', 'Serie histórica, valores ajustados y pronósticos para enero 2022 a diciembre 2024.'],
    ['6. Promedio 2022', round(promedios_anuales.iloc[0], 2)],
    ['6. Promedio 2023', round(promedios_anuales.iloc[1], 2)],
    ['6. Promedio 2024', round(promedios_anuales.iloc[2], 2)],
    ['6. Interpretación', 'Los promedios anuales aumentan; diciembre presenta picos estacionales.'],
    ['7. Mejor método', mejor],
    ['7. Explicación', 'Tuvo el menor RMSE y MAPE en los 24 meses de prueba (2020-2021).'],
    ['7. Limitación', 'Los resultados son pronósticos y no representan salarios observados de 2022-2024.']
], columns=['Pregunta', 'Respuesta'])
with pd.ExcelWriter('Resultados_ARIMA_Salarios.xlsx') as writer:
    ts.to_excel(writer, sheet_name='Datos históricos')
    pronosticos.to_excel(writer, sheet_name='Pronósticos 2022-2024')
    comparacion.to_excel(writer, sheet_name='Métricas')
    validacion.to_excel(writer, sheet_name='Validación')
    coeficientes.to_excel(writer, sheet_name='Coeficientes SARIMA')
    respuestas.to_excel(writer, sheet_name='Respuestas', index=False)

from google.colab import files
files.download('Resultados_ARIMA_Salarios.xlsx')
